# CReM vs oCReM Consistency Tutorial

This tutorial compares whether CReM and oCReM generate the same structures for the same inputs.

We only compare generated result sets in this notebook.

## Install CReM

Install `crem` in the current Python environment before running the comparison notebook.

```bash
conda activate ocrem
pip install crem
```

If you are using another environment name, replace `ocrem` with your actual environment name.

## Database

This notebook is intended to compare CReM and oCReM with the ChEMBL36 database.

Replace the placeholder URLs below with the actual download links you want to use.

```bash
# oCReM ChEMBL36 database
curl -L "https://zenodo.org/records/19107922/files/ocrem_chembl_36_sqlite.tar.gz" -o chembl_36.tar.gz
tar -xzf chembl_36.tar.gz

# CReM ChEMBL36 database
curl -L "https://zenodo.org/records/19393448/files/crem_chembl36.tar.gz" -o crem_chembl36.tar.gz
tar -xzf crem_chembl36.tar.gz
```

After extraction, make sure the database files are available in the current working directory

In [ ]:
ocrem_db_path = "chembl_36.db"
crem_db_path = "crem_chembl36.db"


## Mutate Molecule <a name='mutate'></a>

Compare the result set produced by `mutate_mol` directly.

In [2]:
import os
import sys

import pandas as pd
from rdkit import Chem

sys.path.append(os.path.dirname(os.getcwd()))

from ta_gen.db import create_db_manager
from ta_gen.ocrem.ocrem import mutate_mol as ocrem_mutate_mol
from crem.crem import mutate_mol as crem_mutate_mol

m = Chem.MolFromSmiles("c1(c(cccc1Cl)Cl)C(=O)Nc1ccnc(c1)NC(=O)C")

# ocrem
db_manager = create_db_manager("sqlite", db_path=ocrem_db_path)
ocrem_raw_results = list(ocrem_mutate_mol(m, db_manager, max_inc=1, radius=3))
ocrem_results = sorted(set(ocrem_raw_results))
# crem
crem_raw_results = list(crem_mutate_mol(m, crem_db_path, max_inc=1, radius=3))
crem_results = sorted(set(crem_raw_results))

mutate_common = sorted(set(ocrem_results) & set(crem_results))
mutate_ocrem_only = sorted(set(ocrem_results) - set(crem_results))
mutate_crem_only = sorted(set(crem_results) - set(ocrem_results))

pd.DataFrame(
    [
        ["oCReM", len(ocrem_results)],
        ["CReM", len(crem_results)],
        ["Intersection", len(mutate_common)],
        ["oCReM only", len(mutate_ocrem_only)],
        ["CReM only", len(mutate_crem_only)],
        ["Exact match", len(mutate_ocrem_only) == 0 and len(mutate_crem_only) == 0],
    ],
    columns=["metric", "value"],
)

,metric,value
0,oCReM,25785
1,CReM,25784
2,Intersection,25783
3,oCReM only,2
4,CReM only,1
5,Exact match,False


## Grow Molecule Comparison <a name='grow'></a>

Compare the result set produced by `grow_mol` directly.

In [3]:
import os
import sys

import pandas as pd
from rdkit import Chem

sys.path.append(os.path.dirname(os.getcwd()))

from ta_gen.db import create_db_manager
from ta_gen.ocrem.ocrem import grow_mol as ocrem_grow_mol
from crem.crem import grow_mol as crem_grow_mol

m = Chem.MolFromSmiles("O=C(C)Oc1ccccc1C(=O)O")

# ocrem
db_manager = create_db_manager("sqlite", db_path=ocrem_db_path)
ocrem_raw_results = list(ocrem_grow_mol(m, db_manager, radius=3))
ocrem_results = sorted(set(ocrem_raw_results))

# crem
crem_raw_results = list(crem_grow_mol(m, crem_db_path, radius=3))
crem_results = sorted(set(crem_raw_results))

grow_common = sorted(set(ocrem_results) & set(crem_results))
grow_ocrem_only = sorted(set(ocrem_results) - set(crem_results))
grow_crem_only = sorted(set(crem_results) - set(ocrem_results))

pd.DataFrame(
    [
        ["oCReM", len(ocrem_results)],
        ["CReM", len(crem_results)],
        ["Intersection", len(grow_common)],
        ["oCReM only", len(grow_ocrem_only)],
        ["CReM only", len(grow_crem_only)],
        ["Exact match", len(grow_ocrem_only) == 0 and len(grow_crem_only) == 0],
    ],
    columns=["metric", "value"],
)

[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors


[14:45:51] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:46:21] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:46:25] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:46:27] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:46:30] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:46:34] WARNING: not removing hydrogen atom with dummy atom neighbors


,metric,value
0,oCReM,135
1,CReM,135
2,Intersection,135
3,oCReM only,0
4,CReM only,0
5,Exact match,True


## Link Molecule Comparison <a name='link'></a>

Compare the result set produced by `link_mols` directly.

In [4]:
import os
import sys

import pandas as pd
from rdkit import Chem

sys.path.append(os.path.dirname(os.getcwd()))

from ta_gen.db import create_db_manager
from ta_gen.ocrem.ocrem import link_mols as ocrem_link_mols
from crem.crem import link_mols as crem_link_mols

m1 = Chem.MolFromSmiles("C(C)Oc1ccccc1C(=O)O")
m2 = Chem.MolFromSmiles("c2ccccc2CNC(C)C(=O)c(cc1)ccc1C")

# ocrem
db_manager = create_db_manager("sqlite", db_path=ocrem_db_path)
ocrem_raw_results = list(ocrem_link_mols(m1, m2, db_manager, radius=1))
ocrem_results = sorted(set(ocrem_raw_results))

# crem
crem_raw_results = list(crem_link_mols(m1, m2, crem_db_path, radius=1))
crem_results = sorted(set(crem_raw_results))

link_common = sorted(set(ocrem_results) & set(crem_results))
link_ocrem_only = sorted(set(ocrem_results) - set(crem_results))
link_crem_only = sorted(set(crem_results) - set(ocrem_results))

pd.DataFrame(
    [
        ["oCReM", len(ocrem_results)],
        ["CReM", len(crem_results)],
        ["Intersection", len(link_common)],
        ["oCReM only", len(link_ocrem_only)],
        ["CReM only", len(link_crem_only)],
        ["Exact match", len(link_ocrem_only) == 0 and len(link_crem_only) == 0],
    ],
    columns=["metric", "value"],
)

[14:46:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:46:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:32] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:32] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:36] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:36] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:40] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:40] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:54] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:47:54] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:48:02] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:48:02] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:48:11] WARNING: not removing hydrogen atom with dummy atom neighbors
[14:48:11] WARNING: not removing hydrogen atom with

,metric,value
0,oCReM,2507
1,CReM,3182
2,Intersection,2490
3,oCReM only,17
4,CReM only,692
5,Exact match,False
